In [1]:
import numpy as np
import pandas as pd
import re

In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error
from sklearn.impute import SimpleImputer

In [3]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("First_comparation")
mlflow.set_tracking_uri("http://localhost:5000")

2025/07/02 12:41:52 INFO mlflow.tracking.fluent: Experiment with name 'First_comparation' does not exist. Creating a new experiment.


In [4]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [ ]:
df_train = pd.read_csv('../data/train/transformed_train.csv')

imputer = SimpleImputer(strategy='mean')  # o 'median', 'most_frequent', etc.
df_train_imputed = imputer.fit_transform(df_train)
df_train = pd.DataFrame(df_train_imputed, columns=df_train.columns)

X_train = df_train.drop(columns=["Precio"])
y_train = df_train["Precio"]

df_val = pd.read_csv('../data/train/transformed_val.csv')

df_val_imputed = imputer.transform(df_val)
df_val = pd.DataFrame(df_val_imputed, columns=df_val.columns)

X_val = df_val.drop(columns=["Precio"])
y_val = df_val["Precio"]

## Modelos

In [24]:
# Hiperparámetros del modelo
params = {
    "subsample": 1.0,
    "reg_lambda": 1,
    "reg_alpha": 0.01,
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.1,
    "gamma": 0,
    "colsample_bytree": 0.6
}

def predict(X, model1, model2, mask):
    """
    Predict prices using two different models based on the 'Kilómetros' value.

    Args:
        X: pandas DataFrame with features
        model1: Model for new cars (Kilómetros < 1000)
        model2: Model for used cars (Kilómetros >= 1000)

    Returns:
        numpy array of predictions
    """
    preds = np.empty(len(X))

    if mask.any():
        preds[mask] = model1.predict(X[mask])
    if ~mask.any():
        preds[~mask] = model2.predict(X[~mask])
    return preds


class DualModel:
    def __init__(self, base_params=None, threshold_km=1000):
        self.base_params = base_params or {}
        self.threshold_km = threshold_km
        self.model_new = XGBRegressor(**self.base_params)
        self.model_used = XGBRegressor(**self.base_params)

    def fit(self, X, y):
        mask_new = X["Kilómetros"] < self.threshold_km
        mask_used = ~mask_new

        self.model_new.fit(X[mask_new], y[mask_new])
        self.model_used.fit(X[mask_used], y[mask_used])

    def predict(self, X):
        mask_new = X["Kilómetros"] < self.threshold_km
        mask_used = ~mask_new

        preds = np.empty(len(X))
        if mask_new.any():
            preds[mask_new] = self.model_new.predict(X[mask_new])
        if mask_used.any():
            preds[mask_used] = self.model_used.predict(X[mask_used])
        return preds
    
    def log_to_mlflow(self, signature, input_example):
        mlflow.sklearn.log_model(
            sk_model=self.model_new,
            name="model_new",
            signature=signature,
            input_example=input_example
        )

        mlflow.sklearn.log_model(
            sk_model=self.model_used,
            name="model_used",
            signature=signature,
            input_example=input_example
        )

In [25]:
params = {
    "subsample": 1.0,
    "reg_lambda": 1,
    "reg_alpha": 0.01,
    "n_estimators": 1000,
    "max_depth": 6,
    "learning_rate": 0.1,
    "gamma": 0,
    "colsample_bytree": 0.6
}

dual_model = DualModel(base_params=params)

with mlflow.start_run():
    dual_model.fit(X_train, y_train)

    # Predicción
    y_pred_train = dual_model.predict(X_train)
    y_pred_val = dual_model.predict(X_val)

    # Métricas
    metrics = calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

    # Log en MLflow
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    dual_model.log_to_mlflow(signature=infer_signature(X_train, y_pred_train),input_example=X_train.iloc[:5])
    


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.51e+12   |   3.73e+13
R²           |   0.9871     |   0.9175
MAE          |   5.89e+05   |   2.55e+06
EVS          |   0.9871     |   0.9175
Max Error    |   2.10e+08   |   1.98e+08
Median AE    |   3.36e+05   |   1.29e+06

🏃 View run flawless-fly-986 at: http://localhost:5000/#/experiments/982858775142351238/runs/6a5b3a64711f44a59f1ee97338d95358
🧪 View experiment at: http://localhost:5000/#/experiments/982858775142351238


In [26]:
model = XGBRegressor(**params)
with mlflow.start_run():
    model.fit(X_train, y_train)

    # Predicción
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    # Métricas
    metrics = calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

    # Log en MLflow
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=infer_signature(X_train, y_pred_train),
        input_example=X_train.iloc[:5]
    )

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.51e+12   |   3.73e+13
R²           |   0.9871     |   0.9175
MAE          |   5.89e+05   |   2.55e+06
EVS          |   0.9871     |   0.9175
Max Error    |   2.10e+08   |   1.98e+08
Median AE    |   3.36e+05   |   1.29e+06

🏃 View run bustling-swan-639 at: http://localhost:5000/#/experiments/982858775142351238/runs/7502c3d0dc944625a184521ff21c1402
🧪 View experiment at: http://localhost:5000/#/experiments/982858775142351238
